In [ ]:
import pandas as pd
import numpy as np
import os

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
file_path = "/content/drive/MyDrive/RDV_Project/processed_data/taxi_raw_ingested.parquet"

df = pd.read_parquet(file_path)

print(df.shape)
df.head()

(24083384, 7)


,tpep_pickup_datetime,tpep_dropoff_datetime,PULocationID,DOLocationID,trip_distance,fare_amount,month
0,2025-01-01 00:18:38,2025-01-01 00:26:59,229,237,1.60,10.0,2025-01
1,2025-01-01 00:32:40,2025-01-01 00:35:13,236,237,0.50,5.1,2025-01
2,2025-01-01 00:44:04,2025-01-01 00:46:01,141,141,0.60,5.1,2025-01
3,2025-01-01 00:14:27,2025-01-01 00:20:01,244,244,0.52,7.2,2025-01
4,2025-01-01 00:21:34,2025-01-01 00:25:06,244,116,0.66,5.8,2025-01


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24083384 entries, 0 to 24083383
Data columns (total 7 columns):
 #   Column                 Dtype         
---  ------                 -----         
 0   tpep_pickup_datetime   datetime64[us]
 1   tpep_dropoff_datetime  datetime64[us]
 2   PULocationID           int32         
 3   DOLocationID           int32         
 4   trip_distance          float64       
 5   fare_amount            float64       
 6   month                  object        
dtypes: datetime64[us](2), float64(2), int32(2), object(1)
memory usage: 1.1+ GB


In [ ]:
df.describe()

,tpep_pickup_datetime,tpep_dropoff_datetime,PULocationID,DOLocationID,trip_distance,fare_amount
count,24083384,24083384,2.408338e+07,2.408338e+07,2.408338e+07,2.408338e+07
mean,2025-04-05 19:50:17.265199,2025-04-05 20:06:46.641920,1.622214e+02,1.617849e+02,6.828011e+00,1.788535e+01
min,2007-12-05 18:45:00,2007-12-05 19:02:00,1.000000e+00,1.000000e+00,0.000000e+00,-1.807600e+03
25%,2025-02-20 22:31:47.750000,2025-02-20 22:46:00,1.160000e+02,1.120000e+02,1.020000e+00,8.600000e+00
50%,2025-04-07 10:24:38,2025-04-07 10:42:52,1.610000e+02,1.620000e+02,1.800000e+00,1.350000e+01
75%,2025-05-19 21:15:42,2025-05-19 21:30:25.250000,2.330000e+02,2.340000e+02,3.490000e+00,2.184000e+01
max,2025-06-30 23:59:59,2025-07-01 22:36:42,2.650000e+02,2.650000e+02,3.860884e+05,8.633721e+05
std,NaN,NaN,6.597955e+01,7.020774e+01,6.305676e+02,1.910936e+02


In [ ]:
missing_values = df.isnull().sum()

print(missing_values)

tpep_pickup_datetime     0
tpep_dropoff_datetime    0
PULocationID             0
DOLocationID             0
trip_distance            0
fare_amount              0
month                    0
dtype: int64


In [ ]:
df = df.dropna()

print(df.shape)

(24083384, 7)


In [ ]:
df["tpep_pickup_datetime"] = pd.to_datetime(
    df["tpep_pickup_datetime"]
)

df["tpep_dropoff_datetime"] = pd.to_datetime(
    df["tpep_dropoff_datetime"]
)

In [ ]:
df.dtypes

,0
tpep_pickup_datetime,datetime64[us]
tpep_dropoff_datetime,datetime64[us]
PULocationID,int32
DOLocationID,int32
trip_distance,float64
fare_amount,float64
month,object


In [ ]:
df["trip_duration"] = (
    df["tpep_dropoff_datetime"]
    - df["tpep_pickup_datetime"]
).dt.total_seconds() / 60

In [ ]:
df["pickup_hour"] = (
    df["tpep_pickup_datetime"]
    .dt.hour
)

In [ ]:
def categorize_time(hour):
    if 5 <= hour < 12:
        return "Pagi"
    elif 12 <= hour < 18:
        return "Siang"
    else:
        return "Malam"

df["time_category"] = (
    df["pickup_hour"]
    .apply(categorize_time)
)

In [ ]:
df["time_category"].value_counts()

,count
time_category,
Malam,10557841
Siang,8297570
Pagi,5227973


In [ ]:
df = df[
    df["trip_distance"] > 0
]

In [ ]:
df = df[
    df["fare_amount"] > 0
]

In [ ]:
df = df[
    (df["trip_duration"] > 0)
    &
    (df["trip_duration"] <= 300)
]

In [ ]:
df = df[
    (df["PULocationID"] > 0)
    &
    (df["DOLocationID"] > 0)
]

In [ ]:
def categorize_trip(distance):
    if distance < 3:
        return "Pendek"
    elif distance < 10:
        return "Sedang"
    else:
        return "Jauh"

df["trip_category"] = (
    df["trip_distance"]
    .apply(categorize_trip)
)

In [ ]:
print(df.shape)

df.head()

(22012724, 11)


,tpep_pickup_datetime,tpep_dropoff_datetime,PULocationID,DOLocationID,trip_distance,fare_amount,month,trip_duration,pickup_hour,time_category,trip_category
0,2025-01-01 00:18:38,2025-01-01 00:26:59,229,237,1.60,10.0,2025-01,8.350000,0,Malam,Pendek
1,2025-01-01 00:32:40,2025-01-01 00:35:13,236,237,0.50,5.1,2025-01,2.550000,0,Malam,Pendek
2,2025-01-01 00:44:04,2025-01-01 00:46:01,141,141,0.60,5.1,2025-01,1.950000,0,Malam,Pendek
3,2025-01-01 00:14:27,2025-01-01 00:20:01,244,244,0.52,7.2,2025-01,5.566667,0,Malam,Pendek
4,2025-01-01 00:21:34,2025-01-01 00:25:06,244,116,0.66,5.8,2025-01,3.533333,0,Malam,Pendek


In [ ]:
df.describe()

,tpep_pickup_datetime,tpep_dropoff_datetime,PULocationID,DOLocationID,trip_distance,fare_amount,trip_duration,pickup_hour
count,22012724,22012724,2.201272e+07,2.201272e+07,2.201272e+07,2.201272e+07,2.201272e+07,2.201272e+07
mean,2025-04-04 23:11:57.296950,2025-04-04 23:28:12.803920,1.627793e+02,1.623248e+02,6.510506e+00,1.941099e+01,1.625845e+01,1.426401e+01
min,2007-12-05 18:45:00,2007-12-05 19:02:00,1.000000e+00,1.000000e+00,1.000000e-02,1.000000e-02,1.666667e-02,0.000000e+00
25%,2025-02-20 09:16:44.750000,2025-02-20 09:32:07.750000,1.250000e+02,1.130000e+02,1.090000e+00,9.300000e+00,7.966667e+00,1.000000e+01
50%,2025-04-05 21:48:25,2025-04-05 22:03:15.500000,1.610000e+02,1.620000e+02,1.830000e+00,1.420000e+01,1.285000e+01,1.500000e+01
75%,2025-05-18 16:26:03,2025-05-18 16:50:54,2.330000e+02,2.340000e+02,3.520000e+00,2.249000e+01,2.023333e+01,1.900000e+01
max,2025-06-30 23:59:59,2025-07-01 02:08:18,2.650000e+02,2.650000e+02,3.860884e+05,8.633721e+05,2.994667e+02,2.300000e+01
std,NaN,NaN,6.559406e+01,7.005844e+01,5.937004e+02,1.996351e+02,1.296015e+01,5.990175e+00


In [ ]:
output_path = "/content/drive/MyDrive/RDV_Project/processed_data"

cleaned_file = (
    f"{output_path}/taxi_cleaned.parquet"
)

df.to_parquet(
    cleaned_file,
    index=False
)

print("Cleaned data saved")

Cleaned data saved
